# Parte 2a: Modelo A (CNN custom)

**Notebook ejecutada en Colab (con salidas):** [https://colab.research.google.com/drive/1jsKVLagYVW_ujwkjycdLnifEWf6rrMGf?usp=sharing](https://colab.research.google.com/drive/1jsKVLagYVW_ujwkjycdLnifEWf6rrMGf?usp=sharing)

> **Notebook auxiliar.** Se usa para trabajar en paralelo. Su contenido se integra en `2-train.ipynb`, que es la que se entrega.
> La lógica va en `src/lib/` (`lib.models`, `lib.training`); acá solo se llaman funciones. Al terminar, guarda los embeddings con `save_embeddings` (contrato en `src/lib/evaluation/embedding_store.py`).

**Colab:** se entrena en Google Colab (GPU T4). La primera celda clona el repo, baja las caras alineadas de Drive y deja los checkpoints en Drive, así una sesión cortada se retoma desde la última época. En local la celda solo usa el repo actual.

**Reproducir:** correr todo. Los experimentos ya terminados no se reentrenan: se cargan desde su `last.pt` y se reimprime su log completo; solo se entrenan los que falten.

In [ ]:
# 0. Entorno: Colab (repo clonado + datos de Drive) o local
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/valentinlujambio/tuia-face-recognition-app.git"
    BRANCH = "feature/tp-reconocimiento-facial"
    ROOT = Path("/content/tuia-face-recognition-app")
    DRIVE_DIR = Path("/content/drive/MyDrive/tp1")
    CKPT_ROOT = DRIVE_DIR / "checkpoints"  # una carpeta por experimento

    if ROOT.exists():
        !git -C {ROOT} pull --ff-only
    else:
        !git clone --depth 1 --branch {BRANCH} {REPO_URL} {ROOT}
    %cd {ROOT}
    !pip install -q albumentations==2.0.8 onnx onnxruntime

    if not (ROOT / "data/processed/aligned").exists():
        archive = next(DRIVE_DIR.glob("aligned.*"))
        !mkdir -p data/processed
        if archive.suffix == ".rar":
            !apt-get -qq install -y unrar > /dev/null
            !unrar x -o+ -inul "{archive}" data/processed/
        else:
            !unzip -q -o "{archive}" -d data/processed/
else:
    ROOT = Path.cwd()
    CKPT_ROOT = ROOT / "results" / "custom_cnn" / "checkpoints"

sys.path.insert(0, str(ROOT / "src"))
print("ROOT:", ROOT, "| checkpoints:", CKPT_ROOT)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from lib.dataset.augmentation import build_train_augmentation
from lib.evaluation import EmbeddingSet, results_path, save_embeddings
from lib.models import CustomCNNConfig, CustomFaceCNN, count_parameters
from lib.training import Experiment, FaceDataset, LabelEncoder, load_faces, run_experiment, set_seed
from lib.vision import InputNormalization

CONFIG = json.loads((ROOT / "splits" / "config.json").read_text(encoding="utf-8"))
ALIGNED_ROOT = ROOT / CONFIG["aligned_root"]
SEED = CONFIG["seed"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(SEED)

splits = pd.read_csv(ROOT / "splits" / "splits.csv")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
splits["split"].value_counts()

## 1. Datos
- `train`: entrenamiento (80 identidades). `val_closed`: mismas identidades, para las curvas de loss/accuracy.
- `val_open`: identidades **no vistas**; su AUC de verificación elige el mejor checkpoint y dispara el early stopping, porque la app reconoce personas registradas después del entrenamiento.
- Augmentation de la Parte 1 solo en train; mismo preprocesamiento que la app (`InputNormalization`: RGB y `(x - 127.5) / 127.5`).
- Las ~13k caras de 112×112 entran en RAM (~0,5 GB en uint8), así el DataLoader no lee disco.

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 2

rows = {name: splits[splits["split"] == name].reset_index(drop=True) for name in ("train", "val_closed", "val_open")}
encoder = LabelEncoder(rows["train"]["identity"])
open_encoder = LabelEncoder(rows["val_open"]["identity"])  # solo para saber qué pares son la misma persona
faces = {name: load_faces(ALIGNED_ROOT, df["aligned_path"]) for name, df in rows.items()}

normalization = InputNormalization()
datasets = {
    "train": FaceDataset(faces["train"], encoder.encode(rows["train"]["identity"]), normalization,
                         augment=build_train_augmentation(seed=SEED)),
    "val_closed": FaceDataset(faces["val_closed"], encoder.encode(rows["val_closed"]["identity"]), normalization),
    "val_open": FaceDataset(faces["val_open"], open_encoder.encode(rows["val_open"]["identity"]), normalization),
}
loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                     persistent_workers=NUM_WORKERS > 0)
loaders = {
    "train": DataLoader(datasets["train"], shuffle=True, drop_last=True, **loader_kwargs),
    "val_closed": DataLoader(datasets["val_closed"], shuffle=False, **loader_kwargs),
    "val_open": DataLoader(datasets["val_open"], shuffle=False, **loader_kwargs),
}
# train sin augmentation ni shuffle: centroides para inicializar la cabeza ArcFace en el fine-tuning
train_plain = DataLoader(FaceDataset(faces["train"], datasets["train"].labels, normalization), shuffle=False, **loader_kwargs)
pd.DataFrame({name: {"caras": len(ds), "identidades": len(np.unique(ds.labels))} for name, ds in datasets.items()}).T

## 2. Arquitectura
ResNet chica (`src/lib/models/custom_cnn.py`, importable desde el backend):

| Bloque | Salida | Detalle |
|---|---|---|
| Stem | 32×112×112 | conv 3×3 + BN + PReLU |
| Etapa 1–4 | 32×56², 64×28², 128×14², 256×7² | 2 bloques residuales por etapa, el primero con stride 2 |
| Embedding | 128 | BN → Dropout → global average pooling → FC → BN |

- **Profundidad:** 4 etapas (sugerencia del enunciado, 4–6 bloques) llevan 112 → 7; los atajos residuales permiten 8 bloques sin problemas de gradiente.
- **BatchNorm** tras cada conv: estabiliza el entrenamiento desde cero y tolera lr más altos.
- **Dropout** antes del FC: regulariza con solo ~110 imágenes por identidad.
- **Global pooling:** 256 parámetros de entrada al FC en vez de 7·7·256 = 12.544 (FC de 33k vs 1,6M parámetros); con pocos datos reduce el sobreajuste.
- **Embedding de 128-d** sin normalizar; la normalización L2 la hace quien lo consume (igual que ArcFace en la app).
- **Cabeza** (`src/lib/models/heads.py`, solo para entrenar, no se exporta):
  - `SoftmaxHead` (exp 1 y 2): FC 128 → 80 identidades + cross-entropy.
  - `ArcFaceHead` (exp 3 a 6, como en la clase 4): normaliza embedding y pesos, logit = s·cos(θ + m) para la clase correcta (s = 30, m = 0,5). El margen angular obliga a que cada identidad quede compacta y separada de las demás, que es lo que mide la verificación por coseno.
- **Fine-tuning (exp 4 a 6):** ArcFace desde cero no convergió (exp 3), así que se parte del extractor del exp 1 (`init_from`) y la cabeza ArcFace se inicializa con el centroide normalizado de cada identidad de train. Así la clase correcta ya "gana" desde la primera época y el margen solo compacta lo que softmax ya separó.

In [ ]:
model_config = CustomCNNConfig(channels=(32, 64, 128, 256), blocks_per_stage=2, embedding_dim=128, dropout=0.2)
model = CustomFaceCNN(model_config)
print(model)
print(f"Parámetros del extractor: {count_parameters(model):,}")

## 3. Experimentos e hiperparámetros

Cada experimento es un `Experiment` (`src/lib/training/experiment.py`) y guarda sus checkpoints en `checkpoints/<nombre>`. Lo común a todos:

| | Valor | Por qué |
|---|---|---|
| Pérdida | cross-entropy sobre la cabeza (softmax o ArcFace) | enfoques 1 y 2 del enunciado |
| SGD | momentum 0,9, Nesterov | estándar para CNN desde cero |
| AdamW | como en la clase 4 | se compara contra SGD (exp 1 vs 2) |
| Schedule coseno | warmup lineal desde lr/10 y luego coseno hasta el máximo de épocas | el warmup evita el arranque inestable con lr alto |
| Schedule constante | lr fijo, sin warmup | controles de fine-tuning (exp 5 y 6) |
| Batch size | 128 | entra holgado en una T4 con AMP |
| Early stopping | sobre AUC de verificación en val_open | lo que importa es reconocer personas no vistas |
| AMP | float16 en GPU | ~2x más rápido en T4 |
| Augmentation | la de la Parte 1, solo en train, con `seed` | reproducible |

Los controles 5 y 6 responden si la mejora del exp 4 viene de ArcFace o de entrenar un poco más con lr bajo: los dos parten del exp 1 con lr 0,001 constante, y la única diferencia es el margen (0,5 vs 0). Con margen 0 la cabeza es una softmax normalizada (logit = s·cos θ).

In [ ]:
FINETUNE = dict(head="arcface", lr=0.001, schedule="constant", warmup_epochs=0, epochs=10, patience=10,
                init_from="exp1_softmax_sgd")
EXPERIMENTS = [
    Experiment("exp1_softmax_sgd", "softmax + SGD (baseline)", lr=0.1, weight_decay=5e-4, epochs=60, patience=10),
    Experiment("exp2_softmax_adamw", "softmax + AdamW", optimizer="adamw", lr=1e-3, weight_decay=1e-4,
               epochs=60, patience=10),
    Experiment("exp3_arcface_sgd", "ArcFace desde cero", head="arcface", lr=0.1, epochs=60, patience=15),
    Experiment("exp4_arcface_finetune", "ArcFace fine-tuning desde exp 1, coseno con lr 0,01", head="arcface",
               lr=0.01, warmup_epochs=1, epochs=25, patience=10, init_from="exp1_softmax_sgd"),
    Experiment("exp5_arcface_ft_lr1e-3", "control: ArcFace (m = 0,5) fine-tuning, lr 0,001 constante", **FINETUNE),
    Experiment("exp6_normsoftmax_ft_lr1e-3", "control: igual que exp 5 sin margen (m = 0)", margin=0.0, **FINETUNE),
]
hparams = pd.DataFrame([e.as_row() for e in EXPERIMENTS]).set_index("name").T
hparams.loc["margin"] = [e.margin if e.head == "arcface" else "-" for e in EXPERIMENTS]
hparams.loc["scale"] = [e.scale if e.head == "arcface" else "-" for e in EXPERIMENTS]
hparams.drop("description").fillna("-")

## 4. Entrenamiento
Cada época guarda `last.pt` (estado completo) e `history.json`; `best.pt` cuando mejora el AUC de val_open (marcado con `*`).
Un experimento ya terminado se carga desde `last.pt` y reimprime su log sin reentrenar; uno cortado por Colab sigue desde la última época.

In [ ]:
histories = {}
for exp in EXPERIMENTS:
    run_experiment(exp, model_config, len(encoder), loaders, train_plain, CKPT_ROOT, DEVICE)
    histories[exp.name] = pd.DataFrame(json.loads((CKPT_ROOT / exp.name / "history.json").read_text(encoding="utf-8")))

## 5. Curvas de entrenamiento
Por experimento: loss y accuracy de train vs. validation (val_closed), y AUC/EER de verificación en val_open. La línea punteada marca la mejor época.
Con ArcFace la loss de train incluye el margen y la de validación no (s·cos θ), así que no son comparables en escala con las de softmax.

In [ ]:
fig, axes = plt.subplots(len(histories), 3, figsize=(16, 3.2 * len(histories)), squeeze=False)
for row, (name, hist) in zip(axes, histories.items()):
    best_epoch = int(hist.loc[hist["val_open_auc"].idxmax(), "epoch"])
    for ax, (metric, title) in zip(row[:2], [("loss", "Loss"), ("acc", "Accuracy")]):
        ax.plot(hist["epoch"], hist[f"train_{metric}"], label="train")
        ax.plot(hist["epoch"], hist[f"val_{metric}"], label="val_closed")
        ax.set_title(f"{name}: {title}")
    row[2].plot(hist["epoch"], hist["val_open_auc"], label="AUC")
    row[2].plot(hist["epoch"], hist["val_open_eer"], label="EER")
    row[2].set_title(f"{name}: verificación en val_open")
    for ax in row:
        ax.axvline(best_epoch, color="gray", linestyle="--", linewidth=1)
        ax.set_xlabel("época")
        ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
for name, hist in histories.items():
    ax.plot(hist["epoch"], hist["val_open_auc"], label=name)
ax.set(title="AUC de verificación en val_open por experimento", xlabel="época", ylabel="AUC")
ax.legend()
plt.show()

## 6. Comparación de experimentos
Mejor época de cada corrida según el AUC en val_open. `test_open` no se usa para elegir nada.

In [ ]:
summary = pd.DataFrame([
    {"experimento": name, "épocas": len(hist), **hist.loc[hist["val_open_auc"].idxmax()].drop(["best", "seconds"])}
    for name, hist in histories.items()
]).set_index("experimento").rename(columns={"epoch": "mejor época"})
BEST_EXPERIMENT = summary["val_open_auc"].idxmax()
print("Elegido:", BEST_EXPERIMENT)
summary.round(4)

**Conclusiones**
- **Exp 1 vs 2 (optimizador):** SGD supera a AdamW. Con AdamW el early stopping cortó con lr ≈ 3,4e-4 y val acc todavía subiendo: no llegó a la fase de lr bajo del coseno, donde SGD logró su mejor época. En los dos el AUC de val_open se estanca en ~0,88–0,89 mientras val_closed sigue mejorando: softmax separa las 80 identidades de train pero generaliza menos a personas nuevas.
- **Sobreajuste (exp 1):** train acc 0,99 vs val_closed 0,91.
- **Exp 3 (ArcFace desde cero):** no convergió. La loss queda trabada en ~11,4 y el AUC ≈ 0,5 (azar): desde cero todos los ángulos están cerca de 90° y con m = 0,5 la clase correcta nunca supera a las demás. Falla conocida de ArcFace con pocos datos y sin preentrenamiento.
- **Exp 4 (ArcFace fine-tuning, coseno con lr 0,01):** su mejor época es la 0 (warmup, lr 0,001). Al subir a lr 0,01 el AUC cae y no se recupera: lr demasiado alto para fine-tuning.
- **Exp 5 y 6 (controles):** CONTROLES_PENDIENTES

## 7. Exportación
Se exporta el `best.pt` del experimento elegido (`BEST_EXPERIMENT`, mayor AUC en val_open) con `export_onnx` (`src/lib/models/export.py`): batch dinámico, salida `(N, 128)` sin normalizar, y verificación de que onnxruntime da lo mismo que torch.
`models/custom_cnn.onnx` no se versiona (`*.onnx` en `.gitignore`): se copia a Drive para compartirlo con link público y ponerlo en el README.
En la app: `MODEL_NAME=custom_cnn.onnx`, `EMBEDDING_DIM=128`, mismo preprocesamiento (`InputNormalization`) y normalización L2.

In [ ]:
import shutil

from lib.evaluation.cost import file_size_mb, ms_per_face, onnx_parameters
from lib.models import export_onnx
from lib.vision import OnnxFaceEmbedder

best_model = CustomFaceCNN.from_checkpoint(CKPT_ROOT / BEST_EXPERIMENT / "best.pt")
onnx_path = export_onnx(best_model, ROOT / "models" / "custom_cnn.onnx")
if IN_COLAB:
    shutil.copy(onnx_path, DRIVE_DIR / onnx_path.name)

# Mismo camino que la app: .onnx + InputNormalization + L2.
embedder = OnnxFaceEmbedder(onnx_path, normalization=normalization)
# Costo con lib.evaluation.cost, igual que el Modelo B. Es una referencia en la CPU de Colab: la comparación
# de tiempos entre modelos se hace en 2-train.ipynb, midiendo los dos en la misma máquina y corrida.
pd.Series({
    "experimento": BEST_EXPERIMENT,
    "parámetros": f"{onnx_parameters(onnx_path):,}",
    "tamaño .onnx (MB)": round(file_size_mb(onnx_path), 2),
    "ms por cara (CPU Colab, batch 1)": round(ms_per_face(embedder, list(faces["val_open"][:200])), 2),
    "dimensión del embedding": best_model.embedding_dim,
})

## 8. Embeddings para la evaluación
Se calculan sobre todos los splits salvo `train` y se guardan con el contrato común.

In [ ]:
eval_rows = splits[splits["split"] != "train"].reset_index(drop=True)
embeddings = embedder.embed(load_faces(ALIGNED_ROOT, eval_rows["aligned_path"]))

npz_path = save_embeddings(
    EmbeddingSet(
        model="custom_cnn",
        paths=eval_rows["aligned_path"].to_numpy(),
        labels=eval_rows["identity"].to_numpy(),
        splits=eval_rows["split"].to_numpy(),
        embeddings=embeddings,
    ),
    results_path(ROOT, "custom_cnn"),
)
if IN_COLAB:
    shutil.copy(npz_path, DRIVE_DIR / "embeddings_custom_cnn.npz")
print(npz_path, embeddings.shape)
eval_rows["split"].value_counts()